In [17]:
import os
import json

from importlib.metadata import version

%load_ext autoreload
%autoreload 2

from bpe_open_ai_gpt2 import Encoder, get_pairs

print("tiktoken version: ", version("tiktoken"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
tiktoken version:  0.12.0


In [2]:
text=("Transthyretin Amyloid Cardiomyopathy (ATTR-CM):  A fatal, underdiagnosed condition caused "
      "by abnormal protein buildup that stiffens the heart muscle, typically affecting older adult.")

### BPE from tiktoken

In [3]:
import tiktoken

tik_tokenizer=tiktoken.get_encoding("gpt2")
integers=tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
strings=tik_tokenizer.decode(integers)
print(strings)
tik_tokenizer.n_vocab

[51, 2596, 301, 12114, 1186, 259, 1703, 2645, 1868, 5172, 72, 9145, 27189, 357, 1404, 5446, 12, 24187, 2599, 220, 317, 10800, 11, 739, 47356, 1335, 4006, 4073, 416, 18801, 7532, 40502, 326, 15175, 641, 262, 2612, 8280, 11, 6032, 13891, 4697, 4044, 13]
Transthyretin Amyloid Cardiomyopathy (ATTR-CM):  A fatal, underdiagnosed condition caused by abnormal protein buildup that stiffens the heart muscle, typically affecting older adult.


50257

In [4]:
for i in integers: print(tik_tokenizer.decode([i]), end=', ')

T, ran, st, hy, ret, in,  Am, yl, oid,  Card, i, omy, opathy,  (, AT, TR, -, CM, ):,  ,  A,  fatal, ,,  under, diagn, osed,  condition,  caused,  by,  abnormal,  protein,  buildup,  that,  stiff, ens,  the,  heart,  muscle, ,,  typically,  affecting,  older,  adult, ., 

### BPE from the original implementation in GPT-2

In [5]:
main_dir='/Users/reaungamornrat.sureerat/data/BPE/pracetice'
gpt2_dir=f"{main_dir}/gpt2_bpe"
encode_path=f"{gpt2_dir}/encoder.json" # vocab file
bpe_merges_path=f"{gpt2_dir}/vocab.bpe" # bpe merges/ranks

with open(encode_path, "r") as f: encoder=json.load(f)
with open(bpe_merges_path, 'r', encoding='utf-8') as f: bpe_data=f.read()
bpe_merges=[tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]
orig_tokenizer=Encoder(encoder=encoder, bpe_merges=bpe_merges)

integers=orig_tokenizer.encode(text)
print(integers)

strings=orig_tokenizer.decode(integers)
print(strings)

print(text==strings)
for ref, obs in zip(tuple(text), tuple(strings)):
    if ref!=obs: print(f"{ref==obs}, '{ref}', '{obs}'")

[51, 2596, 301, 12114, 1186, 259, 1703, 2645, 1868, 5172, 72, 9145, 27189, 357, 1404, 5446, 12, 24187, 2599, 220, 317, 10800, 11, 739, 47356, 1335, 4006, 4073, 416, 18801, 7532, 40502, 326, 15175, 641, 262, 2612, 8280, 11, 6032, 13891, 4697, 4044, 13]
Transthyretin Amyloid Cardiomyopathy (ATTR-CM):  A fatal, underdiagnosed condition caused by abnormal protein buildup that stiffens the heart muscle, typically affecting older adult.
True


In [6]:
for i in integers: print(tik_tokenizer.decode([i]), end=', ')

T, ran, st, hy, ret, in,  Am, yl, oid,  Card, i, omy, opathy,  (, AT, TR, -, CM, ):,  ,  A,  fatal, ,,  under, diagn, osed,  condition,  caused,  by,  abnormal,  protein,  buildup,  that,  stiff, ens,  the,  heart,  muscle, ,,  typically,  affecting,  older,  adult, ., 

### BPE via Hugging Face transformers

In [7]:
import transformers

transformers.__version__

/Users/reaungamornrat.sureerat/miniforge3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'5.12.0'

In [14]:
from transformers import GPT2Tokenizer
hf_tokenizer=GPT2Tokenizer.from_pretrained("gpt2")
print(hf_tokenizer(strings)["input_ids"])

[51, 2596, 301, 12114, 1186, 259, 1703, 2645, 1868, 5172, 72, 9145, 27189, 357, 1404, 5446, 12, 24187, 2599, 220, 317, 10800, 11, 739, 47356, 1335, 4006, 4073, 416, 18801, 7532, 40502, 326, 15175, 641, 262, 2612, 8280, 11, 6032, 13891, 4697, 4044, 13]


In [15]:
from transformers import GPT2TokenizerFast
hf_tokenizer_fast=GPT2TokenizerFast.from_pretrained("gpt2")
print(hf_tokenizer(strings)["input_ids"])

[51, 2596, 301, 12114, 1186, 259, 1703, 2645, 1868, 5172, 72, 9145, 27189, 357, 1404, 5446, 12, 24187, 2599, 220, 317, 10800, 11, 739, 47356, 1335, 4006, 4073, 416, 18801, 7532, 40502, 326, 15175, 641, 262, 2612, 8280, 11, 6032, 13891, 4697, 4044, 13]


In [19]:
fpath='../the-verdict.txt'
with open(fpath, 'r', encoding='utf-8') as f: raw_text=f.read()

In [20]:
%timeit orig_tokenizer.encode(raw_text) # original openai GPT-2

3.23 ms ± 5.04 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
%timeit tik_tokenizer.encode(raw_text) # tiktoken

595 μs ± 13.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [22]:
%timeit hf_tokenizer(raw_text)['input_ids']

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


2.68 ms ± 6.12 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)['input_ids']

2.69 ms ± 4.01 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [24]:
%timeit hf_tokenizer_fast(raw_text)['input_ids']

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


2.68 ms ± 9.56 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)['input_ids']

2.64 ms ± 7.15 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
